# 05 — Groundsource: Merge and Exploratory Data Analysis

**Purpose:** Merge positive flood samples (is_flood = 1) with generated no-flood samples
(is_flood = 0) into a single machine-learning-ready dataset. Then characterise the combined
dataset through visualisations and a printed analytical summary.

**Column schema:** Columns are reordered into a logical sequence:
1. Identifiers: `uuid`, `event_id`, `start_date`, `end_date`, `duration_days`
2. Geometry and land cover: `area_km2_original`, `area_km2_mollweide`, `geometry`,
   `urban_built_up_area_m2`, `polygon_total_area_m2`, `urban_percentage`
3. Pluvial indices: `upa_max`, `upa_p95`, `upa_p99`, `PFDI_max`, `PFDI_p95`, `PFDI_p99`
4. Raw IMERG data: `imerg_type`, `imerg_meta`, `imerg_mask`, `imerg_matrix`
5. Intensity features: `30_max_rainfall_intens` … `1440_max_rainfall_intens`
6. Label: `is_flood`

**Input:**
- `outputs/groundsource_rain_master.parquet` — enriched flood events (from 03b)
- `outputs/no_flood_batches/no_flood_batch_*.parquet` — no-flood events (from 04)

**Output:**
- `outputs/groundsource_unified_ml_dataset.parquet` — final ML-ready dataset
- Key charts and a printed analytical summary

In [ ]:
import glob
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

In [ ]:
# --- CONFIGURATION ---

FLOOD_PATH     = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_rain_master.parquet"
NO_FLOOD_DIR   = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\no_flood_batches"
OUTPUT_PATH    = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_unified_ml_dataset.parquet"

URBAN_THRESHOLD = 40   # % — threshold for the urban-focused comparison in EDA
DURATIONS_MIN   = [30, 60, 120, 240, 360, 720, 1440]

# Logical column order for the final dataset
COLUMN_ORDER = [
    # Identifiers
    'uuid', 'event_id', 'start_date', 'end_date', 'duration_days',
    # Geometry and land cover
    'area_km2_original', 'area_km2_mollweide', 'geometry',
    'urban_built_up_area_m2', 'polygon_total_area_m2', 'urban_percentage',
    # Pluvial indices
    'upa_max', 'upa_p95', 'upa_p99', 'PFDI_max', 'PFDI_p95', 'PFDI_p99',
    # Raw IMERG
    'imerg_type', 'imerg_meta', 'imerg_mask', 'imerg_matrix',
    # Intensity features
    '30_max_rainfall_intens', '60_max_rainfall_intens', '120_max_rainfall_intens',
    '240_max_rainfall_intens', '360_max_rainfall_intens', '720_max_rainfall_intens',
    '1440_max_rainfall_intens',
    # Label (always last)
    'is_flood',
]

## 1. Load positive samples (flood events)

In [ ]:
t_start = time.time()

flood_df = pd.read_parquet(FLOOD_PATH)
flood_df['is_flood'] = 1
flood_df['start_date'] = pd.to_datetime(flood_df['start_date'])
flood_df['end_date']   = pd.to_datetime(flood_df['end_date'])
flood_df['duration_days'] = (flood_df['end_date'] - flood_df['start_date']).dt.days + 1

# Assign event_id if not present
if 'event_id' not in flood_df.columns:
    flood_df['event_id'] = flood_df['uuid']

print(f"Flood events loaded    : {len(flood_df):,}")

## 2. Load negative samples (no-flood events)

In [ ]:
no_flood_files = sorted(glob.glob(f"{NO_FLOOD_DIR}/no_flood_batch_*.parquet"))
print(f"No-flood batch files found: {len(no_flood_files)}")

if no_flood_files:
    noflood_df = pd.concat([pd.read_parquet(f) for f in no_flood_files], ignore_index=True)
    noflood_df['start_date'] = pd.to_datetime(noflood_df['start_date'])
    noflood_df['end_date']   = pd.to_datetime(noflood_df['end_date'])
    noflood_df['duration_days'] = (
        (noflood_df['end_date'] - noflood_df['start_date']).dt.days + 1
    )
    # Drop helper column not needed in final dataset
    noflood_df = noflood_df.drop(columns=['source_uuid'], errors='ignore')
    print(f"No-flood events loaded : {len(noflood_df):,}")
else:
    print("WARNING: No no-flood batch files found. Final dataset will contain flood events only.")
    noflood_df = pd.DataFrame()

## 3. Merge and reorder columns

In [ ]:
if len(noflood_df) > 0:
    # Keep only columns present in both DataFrames to avoid alignment issues
    common_cols = [c for c in flood_df.columns if c in noflood_df.columns]
    combined = pd.concat(
        [flood_df[common_cols], noflood_df[common_cols]],
        ignore_index=True
    )
else:
    combined = flood_df.copy()

# Apply logical column order, keeping any extra columns at the end
ordered = [c for c in COLUMN_ORDER if c in combined.columns]
extra   = [c for c in combined.columns if c not in ordered]
combined = combined[ordered + extra]

print(f"Combined dataset shape : {combined.shape}")
print(f"is_flood breakdown     :")
print(combined['is_flood'].value_counts())

## 4. Exploratory Data Analysis

In [ ]:
# --- 4a. Class balance ---

class_counts = combined['is_flood'].value_counts().sort_index()
labels = ['No Flood (0)', 'Flood (1)']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution — Groundsource Unified Dataset', fontsize=13)

axes[0].bar(labels, class_counts.values, color=['#e07b54', '#4878d0'], edgecolor='white')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + max(class_counts.values) * 0.01,
                 f'{v:,}', ha='center', fontsize=11)
axes[0].set_ylabel('Count')
axes[0].set_title('Event counts by class')

axes[1].pie(
    class_counts.values,
    labels=[f"{l}\n{v/len(combined)*100:.1f}%" for l, v in zip(labels, class_counts.values)],
    colors=['#e07b54', '#4878d0'],
    startangle=90,
    wedgeprops={'edgecolor': 'white'}
)
axes[1].set_title('Class balance')

plt.tight_layout()
plt.show()

In [ ]:
# --- 4b. Rainfall intensity distributions by class (1-hr and 24-hr) ---

if len(noflood_df) > 0:
    flood_vals  = combined[combined['is_flood'] == 1]
    nflood_vals = combined[combined['is_flood'] == 0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Rainfall Intensity by Class — Groundsource', fontsize=13)

    for ax, col, title in [
        (axes[0], '60_max_rainfall_intens',   '1-hour peak intensity'),
        (axes[1], '1440_max_rainfall_intens', '24-hour peak intensity'),
    ]:
        bins = np.linspace(0, np.percentile(combined[col].dropna(), 99), 60)
        ax.hist(flood_vals[col].dropna(),  bins=bins, alpha=0.6,
                color='#4878d0', label='Flood',    density=True)
        ax.hist(nflood_vals[col].dropna(), bins=bins, alpha=0.6,
                color='#e07b54', label='No-Flood', density=True)
        ax.set_xlabel('mm/hr')
        ax.set_ylabel('Density')
        ax.set_title(title)
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# --- 4c. Intensity–duration curves by class ---

intens_cols = [f"{d}_max_rainfall_intens" for d in DURATIONS_MIN]

fig, ax = plt.subplots(figsize=(10, 6))

if len(noflood_df) > 0:
    for label, subset, color in [
        ('Flood',    combined[combined['is_flood'] == 1], '#4878d0'),
        ('No-Flood', combined[combined['is_flood'] == 0], '#e07b54'),
    ]:
        medians = [subset[c].median() for c in intens_cols]
        ax.plot(DURATIONS_MIN, medians, 'o-', color=color, label=label, linewidth=2)
else:
    medians = [combined[c].median() for c in intens_cols]
    ax.plot(DURATIONS_MIN, medians, 'o-', color='#4878d0', label='Flood', linewidth=2)

ax.set_xscale('log')
ax.set_xlabel('Duration (minutes, log scale)')
ax.set_ylabel('Median peak intensity (mm/hr)')
ax.set_title('Intensity–Duration Curve by Class — Groundsource')
ax.set_xticks(DURATIONS_MIN)
ax.set_xticklabels([str(d) for d in DURATIONS_MIN])
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 4d. Urban basin comparison (events with ≥ URBAN_THRESHOLD% built-up) ---

urban = combined[combined['urban_percentage'] >= URBAN_THRESHOLD].copy()
print(f"Urban events (≥{URBAN_THRESHOLD}%): {len(urban):,}")

compare_cols = ['60_max_rainfall_intens', '1440_max_rainfall_intens', 'PFDI_max', 'upa_max']
col_labels   = ['60-min intensity', '1440-min intensity', 'PFDI_max', 'UPA max']

if len(noflood_df) > 0 and (urban['is_flood'] == 0).any():
    flood_urban   = urban[urban['is_flood'] == 1]
    nflood_urban  = urban[urban['is_flood'] == 0]

    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    fig.suptitle(f'Urban Basin Comparison (≥{URBAN_THRESHOLD}% built-up) — Groundsource', fontsize=12)

    for ax, col, lbl in zip(axes, compare_cols, col_labels):
        data = [flood_urban[col].dropna().values, nflood_urban[col].dropna().values]
        ax.boxplot(data, labels=['Flood', 'No-Flood'], showfliers=False)
        ax.set_title(lbl)
        ax.set_ylabel('Value')
        if col in ('PFDI_max', 'upa_max'):
            ax.set_yscale('log')

    plt.tight_layout()
    plt.show()

## 5. Save final dataset

In [ ]:
combined.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(combined):,} rows × {len(combined.columns)} columns → {OUTPUT_PATH}")
print(f"Total runtime: {time.time()-t_start:.1f}s")

## 6. Analytical summary

In [ ]:
flood_rows   = combined[combined['is_flood'] == 1]
noflood_rows = combined[combined['is_flood'] == 0]

print("=" * 65)
print("  GROUNDSOURCE — UNIFIED ML DATASET SUMMARY")
print("=" * 65)
print(f"  Total records          : {len(combined):>12,}")
print(f"  Flood events (1)       : {len(flood_rows):>12,}  "
      f"({len(flood_rows)/len(combined)*100:.1f}%)")
print(f"  No-flood events (0)    : {len(noflood_rows):>12,}  "
      f"({len(noflood_rows)/len(combined)*100:.1f}%)")
print(f"  Total columns          : {len(combined.columns):>12}")
print(f"  Date range             : {combined['start_date'].min().date()} → "
      f"{combined['start_date'].max().date()}")
print()
print("  FLOOD EVENTS — feature statistics:")
print(f"    Median area (mollweide) : {flood_rows['area_km2_mollweide'].median():.3f} km²")
print(f"    Median urban %          : {flood_rows['urban_percentage'].median():.1f}%")
print(f"    Median PFDI_max         : {flood_rows['PFDI_max'].median():.3f}")
print(f"    Pluvial fraction        : {(flood_rows['PFDI_max'] <= 1).mean()*100:.1f}%")
print(f"    Median 60-min intensity : {flood_rows['60_max_rainfall_intens'].median():.3f} mm/hr")
print(f"    Median 24-hr intensity  : {flood_rows['1440_max_rainfall_intens'].median():.3f} mm/hr")

if len(noflood_rows) > 0:
    print()
    print("  NO-FLOOD EVENTS — feature statistics:")
    print(f"    Median 60-min intensity : {noflood_rows['60_max_rainfall_intens'].median():.3f} mm/hr")
    print(f"    Median 24-hr intensity  : {noflood_rows['1440_max_rainfall_intens'].median():.3f} mm/hr")

print("=" * 65)